In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")


landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")


from datetime import datetime
print("=" * 80)
print(f"[STEP 1] Bronze -> Silver run started at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print("Received parameters:")
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'" + ("  (empty -> will default to FULL)" if not load_type.strip() else ""))

if not table_name.strip():
    print("  WARNING: table_name is empty - the bronze read will fail.")
if not landing_timestamp.strip():
    print("  WARNING: landing_timestamp is empty - the bronze filter will return no rows.")

In [0]:
source_table = f"adb_neeleshcaresync.bronze.{table_name}"
filter_condition = f"landing_stamp = '{landing_timestamp}'"

print("-" * 80)
print("[STEP 2] Reading data from bronze")
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print("  Equivalent SQL query:")
print(f"    SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"  Columns read ({len(bronze_df.columns)}): {bronze_df.columns}")
print(f"  Rows matched for landing_stamp '{landing_timestamp}': {bronze_count}")
if bronze_count == 0:
    print("  WARNING: No rows found in bronze for this landing timestamp.")

In [0]:
from pyspark.sql import functions as F
print("-" * 80)
print("[STEP 3] Adding audit columns: insert_timestamp, update_timestamp = current_timestamp()")
updated_df=bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp",F.current_timestamp())
print("  Equivalent SQL query:")
print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
print(f"    FROM {source_table} WHERE {filter_condition}")
print(f"  Resulting columns ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
from pyspark.sql import functions as F
print("-" * 80)
print("[STEP 4] Re-applying audit columns (same logic as STEP 3)")
updated_df=bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp",F.current_timestamp())
print(f"  Resulting columns ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_neeleshcaresync.silver.{table_name}"
# Empty load_type defaults to FULL to keep the original behavior
load_type_norm = (load_type or "FULL").strip().upper()

print("-" * 80)
print("[STEP 5] Writing to silver")
print(f"  Target table        : {target_table}")
print(f"  Load type (raw)     : '{load_type}'")
print(f"  Load type (resolved): {load_type_norm}")
target_exists = spark.catalog.tableExists(target_table)
print(f"  Target table exists : {target_exists}")
if target_exists:
    print(f"  Rows in target BEFORE write: {spark.read.table(target_table).count()}")
cols_sql = ", ".join([f"`{c}`" for c in updated_df.columns])

if load_type_norm == "FULL":
    # Full load: overwrite the target table
    print("  Action: FULL load -> overwriting the target table")
    print("  Equivalent SQL query:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS")
    print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
    print(f"    FROM {source_table} WHERE {filter_condition}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed.")

elif load_type_norm == "APPEND":
    # Append: add incoming records to the target table (creates it if missing)
    print("  Action: APPEND -> adding incoming rows to the target table" + ("" if target_exists else " (table will be created)"))
    print("  Equivalent SQL query:")
    print(f"    INSERT INTO {target_table} ({cols_sql})")
    print(f"    SELECT *, current_timestamp(), current_timestamp() FROM {source_table} WHERE {filter_condition}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND completed.")

elif load_type_norm == "MERGE":
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"  Merge keys parsed: {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = 'MERGE'")
    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")

    if not spark.catalog.tableExists(target_table):
        # First run: nothing to merge into, so create the table
        print("  Action: MERGE requested but target does not exist -> creating table with incoming data")
        print("  Equivalent SQL query:")
        print(f"    CREATE TABLE {target_table} AS")
        print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
        print(f"    FROM {source_table} WHERE {filter_condition}")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial table creation completed.")
    else:
        # Keep one row per key to avoid multiple-source-match errors in MERGE
        source_df = updated_df.dropDuplicates(keys)
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # On update, keep the original insert_timestamp and refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}

        print(f"  Source rows before de-duplication on keys: {updated_df.count()}")
        print(f"  Source rows after de-duplication on keys : {source_df.count()}")
        print(f"  Merge condition: {merge_condition}")
        print(f"  Columns updated on match (insert_timestamp preserved): {list(update_set.keys())}")
        set_sql = ",\n      ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        print("  Equivalent SQL query:")
        print(f"    MERGE INTO {target_table} t")
        print(f"    USING (<bronze rows from {source_table} WHERE {filter_condition}, deduplicated on {keys}>) s")
        print(f"    ON {merge_condition}")
        print(f"    WHEN MATCHED THEN UPDATE SET")
        print(f"      {set_sql}")
        print(f"    WHEN NOT MATCHED THEN INSERT *")

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsertAll()
            .execute())

        # Report merge metrics from the latest Delta history entry
        metrics = (DeltaTable.forName(spark, target_table).history(1)
                   .select("operationMetrics").collect()[0]["operationMetrics"])
        print("  MERGE completed. Metrics:")
        for m in ["numSourceRows", "numTargetRowsInserted", "numTargetRowsUpdated", "numTargetRowsDeleted", "numTargetRowsCopied"]:
            if m in metrics:
                print(f"    {m}: {metrics[m]}")

else:
    print(f"  ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE.")

print(f"  Rows in target AFTER write: {spark.read.table(target_table).count()}")

In [0]:
%sql
select * from adb_neeleshcaresync.silver.hospitals

In [0]:
record_count = updated_df.count()
print("-" * 80)
print("[STEP 6] Finishing run")
print(f"  Records processed from bronze: {record_count}")
print(f"  Run finished at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Exiting notebook with value: '{record_count}'")
print("=" * 80)
dbutils.notebook.exit(str(record_count))

In [0]:
%sql
select * from adb_neeleshcaresync.silver.patients